# Medallion refresh
reads `bronze.raw_orders` (this text must not create lineage)


In [ ]:
%pip install great-expectations
%matplotlib inline
import pyspark.sql.functions as F


In [ ]:
run_date = "2026-10-01"
source_table = "bronze.raw_orders"


In [ ]:
raw = spark.table(source_table)
# spark.table("commented.out")
clean = (raw.where(F.col("status") == "paid")
            .select("order_id", "customer_id", (F.col("qty") * F.col("price")).alias("revenue")))
clean.createOrReplaceTempView("clean_orders")


In [ ]:
%%sql
CREATE OR REPLACE TABLE silver.orders_clean AS
SELECT order_id, customer_id, revenue FROM clean_orders


In [ ]:
%%sql
-- orders per customer
INSERT OVERWRITE TABLE gold.customer_revenue
SELECT customer_id, SUM(revenue) AS revenue FROM silver.orders_clean GROUP BY customer_id


In [ ]:
%%scala
spark.table("ignored").show()


In [ ]:
%run Shared/helpers
notebookutils.notebook.run('Validate Orders', 120)
